# Task 3 -- Contrastive Learning and CLIP


In [ ]:
import os
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets
from torch.utils.data import DataLoader
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

try:
    from google.colab import drive
    drive.mount('/content/drive')
    RUN_DIR = '/content/drive/MyDrive/ATML/assignment_00/clip'
except ImportError:
    RUN_DIR = './clip'  # not on Colab

os.makedirs(RUN_DIR, exist_ok=True)
os.makedirs(f'{RUN_DIR}/embeddings', exist_ok=True)
DATA_ROOT = f'{RUN_DIR}/data'

DRY_RUN = True
TEST_SUBSET = 200 if DRY_RUN else None 
PAIR_SAMPLES = 100                       

print(f"RUN_DIR={RUN_DIR} | DRY_RUN={DRY_RUN} | TEST_SUBSET={TEST_SUBSET}")

In [ ]:

%pip install -q ftfy regex tqdm
%pip install -q git+https://github.com/openai/CLIP.git
%pip install -q umap-learn

import clip
import umap
from scipy.linalg import orthogonal_procrustes

In [ ]:

model, preprocess = clip.load("ViT-B/32", device=DEVICE)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"CLIP ViT-B/32 loaded: {n_params:,} parameters")
print(f"Image preprocessing pipeline:\n{preprocess}")

## Task 3.1: Zero-Shot Classification on STL-10

In [ ]:

test_ds_full = datasets.STL10(root=DATA_ROOT, split='test', download=True, transform=preprocess)
classes_stl10 = test_ds_full.classes
print(f"STL-10 classes: {classes_stl10}")
print(f"Full test set size: {len(test_ds_full)}")

if TEST_SUBSET is not None:
    g = torch.Generator().manual_seed(42)
    idx = torch.randperm(len(test_ds_full), generator=g)[:TEST_SUBSET].tolist()
    test_ds = torch.utils.data.Subset(test_ds_full, idx)
else:
    test_ds = test_ds_full

test_loader = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)
print(f"Evaluating on {len(test_ds)} images (DRY_RUN={DRY_RUN})")

In [ ]:
def extract_image_embeddings(model, loader, cache_path=None):
    if cache_path and os.path.exists(cache_path):
        data = torch.load(cache_path)
        return data['embeds'], data['labels']

    model.eval()
    all_embeds, all_labels = [], []
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="encoding images"):
            images = images.to(DEVICE)
            embeds = model.encode_image(images).float()
            all_embeds.append(embeds.cpu())
            all_labels.append(labels)

    all_embeds = torch.cat(all_embeds)
    all_labels = torch.cat(all_labels)
    if cache_path:
        torch.save({'embeds': all_embeds, 'labels': all_labels}, cache_path)
    return all_embeds, all_labels

cache_tag = f"subset{TEST_SUBSET}" if TEST_SUBSET is not None else "full"
image_embeds, image_labels = extract_image_embeddings(
    model, test_loader, cache_path=f'{RUN_DIR}/embeddings/stl10_test_{cache_tag}.pt'
)
print(f"image_embeds shape: {tuple(image_embeds.shape)}")  # (N, 512) for ViT-B/32

In [ ]:

def build_prompts(classnames, strategy):
    if strategy == "plain":
        return [f"{c}" for c in classnames]
    elif strategy == "templated":
        return [f"a photo of a {c}" for c in classnames]
    elif strategy == "descriptive":
        return [f"a high-quality, clear photo of a {c}, prominently visible in the center of the frame" for c in classnames]
    else:
        raise ValueError(strategy)

def zero_shot_accuracy(model, image_embeds, image_labels, classnames, strategy):

    prompts = build_prompts(classnames, strategy)
    tokens = clip.tokenize(prompts).to(DEVICE)

    with torch.no_grad():
        text_embeds = model.encode_text(tokens).float().cpu()

    img_n = F.normalize(image_embeds, dim=-1)
    txt_n = F.normalize(text_embeds, dim=-1)

    similarity = img_n @ txt_n.T                 # (N_images, N_classes)
    predictions = similarity.argmax(dim=-1)
    accuracy = (predictions == image_labels).float().mean().item() * 100
    return accuracy, text_embeds, prompts

strategies = ["plain", "templated", "descriptive"]
results_31 = {}
for strategy in strategies:
    acc, txt_emb, prompts = zero_shot_accuracy(model, image_embeds, image_labels, classes_stl10, strategy)
    results_31[strategy] = {"accuracy": acc, "text_embeds": txt_emb, "prompts": prompts}
    print(f"{strategy:12s} | example prompt: {prompts[0]!r:50s} | accuracy: {acc:.2f}%")

### Your answer

**Analytical questions (Task 3.1(d)):** Compare the accuracies of the three
prompting strategies on the complete test set. Which performed best, and
does that match the "templated prompts are closer to CLIP's training
distribution" explanation above? Are there any classes where a more
descriptive prompt helped or hurt disproportionately?



## Task 3.2: Exploring the Modality Gap

(a) Use the vision and text encoders within CLIP to extract image and label
embeddings for 50-100 STL-10 samples.
(b) Use a dimensionality-reduction technique such as UMAP or t-SNE to
project the embeddings into a two-dimensional space.
(c) Visualize and compare the distributions of the text and image
embeddings.
(d) Briefly explain your findings:
  - How separated are the two modalities?
  - Does normalization affect the modality gap?
  - Why does CLIP still perform well despite this gap?

In [ ]:

g = torch.Generator().manual_seed(42)
pair_idx = torch.randperm(len(image_embeds), generator=g)[:PAIR_SAMPLES]

pair_image_embeds = image_embeds[pair_idx]                       # (n, 512)
pair_labels = image_labels[pair_idx]
templated_text_embeds = results_31["templated"]["text_embeds"]   # (10, 512), one per class
pair_text_embeds = templated_text_embeds[pair_labels]            # (n, 512) -- gather per-sample

print(f"Paired samples: {pair_image_embeds.shape[0]}")
print(f"pair_image_embeds: {tuple(pair_image_embeds.shape)} | pair_text_embeds: {tuple(pair_text_embeds.shape)}")

In [ ]:
def modality_gap_distance(img_embeds, txt_embeds):
    img_centroid = F.normalize(img_embeds, dim=-1).mean(dim=0)
    txt_centroid = F.normalize(txt_embeds, dim=-1).mean(dim=0)
    return (img_centroid - txt_centroid).norm().item()

def plot_modality_gap(ax, img_embeds, txt_embeds, title):
    combined = torch.cat([img_embeds, txt_embeds], dim=0).numpy()
    modality = np.array(["image"] * len(img_embeds) + ["text"] * len(txt_embeds))

    coords = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(combined)

    for m, color in [("image", "tab:blue"), ("text", "tab:orange")]:
        mask = modality == m
        ax.scatter(coords[mask, 0], coords[mask, 1], s=15, alpha=0.6, label=m, color=color)
    ax.set_title(title)
    ax.legend()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

plot_modality_gap(axes[0], pair_image_embeds, pair_text_embeds, "Raw embeddings")
gap_raw = modality_gap_distance(pair_image_embeds, pair_text_embeds)

plot_modality_gap(
    axes[1], F.normalize(pair_image_embeds, dim=-1), F.normalize(pair_text_embeds, dim=-1),
    "L2-normalized embeddings"
)

plt.tight_layout()
plt.show()


print(f"Centroid-distance modality gap (normalized space): {gap_raw:.4f}")

### Your answer

**Analytical questions (Task 3.2(d)):**
- How separated are the two modalities in your plot(s)? Do image and text
  points form clearly distinct regions, or do they interleave?
- Does normalization visibly affect the modality gap in your two panels?
- Given the "relative ranking is all that matters" explanation above, why
  does CLIP still perform well (per your Task 3.1 results) despite this gap?

_(answer here)_

## Task 3.3: Bridging the Modality Gap

In [ ]:

X_pair = F.normalize(pair_image_embeds, dim=-1).numpy()   # image features
Y_pair = F.normalize(pair_text_embeds, dim=-1).numpy()    # text features

R, scale = orthogonal_procrustes(X_pair, Y_pair)
print(f"R shape: {R.shape} | is orthogonal (R^T R ~= I): {np.allclose(R.T @ R, np.eye(R.shape[0]), atol=1e-4)}")


image_embeds_n = F.normalize(image_embeds, dim=-1).numpy()
aligned_image_embeds = torch.from_numpy(image_embeds_n @ R).float()
print(f"aligned_image_embeds shape: {tuple(aligned_image_embeds.shape)}")
print(f"norm preserved by rotation: original={np.linalg.norm(image_embeds_n[0]):.4f}, "
      f"aligned={aligned_image_embeds[0].norm().item():.4f}")

In [ ]:

def zero_shot_accuracy_aligned(aligned_img_embeds, image_labels, text_embeds):
    txt_n = F.normalize(text_embeds, dim=-1)
    similarity = aligned_img_embeds @ txt_n.T   # aligned_img_embeds already unit-norm (rotation preserves norm)
    predictions = similarity.argmax(dim=-1)
    return (predictions == image_labels).float().mean().item() * 100

print(f"{'strategy':12s} {'baseline (Part 0)':>20s} {'aligned':>12s} {'delta':>10s}")
for strategy in strategies:
    txt_emb = results_31[strategy]["text_embeds"]
    acc_aligned = zero_shot_accuracy_aligned(aligned_image_embeds, image_labels, txt_emb)
    acc_baseline = results_31[strategy]["accuracy"]
    print(f"{strategy:12s} {acc_baseline:19.2f}% {acc_aligned:11.2f}% {acc_aligned - acc_baseline:+9.2f}%")

In [ ]:

pair_image_embeds_aligned = aligned_image_embeds[pair_idx]

fig, ax = plt.subplots(figsize=(8, 7))
plot_modality_gap(ax, pair_image_embeds_aligned, F.normalize(pair_text_embeds, dim=-1), "Aligned embeddings (after Procrustes)")
plt.show()

gap_before = modality_gap_distance(pair_image_embeds, pair_text_embeds)
gap_after = modality_gap_distance(pair_image_embeds_aligned, pair_text_embeds)
print(f"Modality gap (centroid distance) before alignment: {gap_before:.4f}")
print(f"Modality gap (centroid distance) after alignment:  {gap_after:.4f}")

### Your answer

**Analytical questions (Task 3.3(e)/(f)):** How does the Procrustes
alignment affect the modality gap, visually and in the quantitative
centroid-distance metric? Does the recomputed zero-shot accuracy improve,
stay flat, or get worse compared to Task 3.1's baseline, for each prompting
strategy -- and does that outcome match what you'd predict from the "only
relative ranking matters for classification" explanation in Task 3.2? If
alignment barely changes accuracy despite visibly shrinking the gap in the
plot, what does that tell you about the relationship between the modality
gap and zero-shot performance?

_(answer here)_